<a href="https://colab.research.google.com/github/fqixiang/workshop_llm_data_collection/blob/main/notebooks/llm_data_collection_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Large Language Models for Data Collection/Annotation in Social Sciences and Humanities


## Package setup (SANE)

In [ ]:
import getpass  # Provides a secure way to handle user passwords or other sensitive input without echoing them on the screen.
import os  # Gives access to operating system functionalities like file paths, environment variables, and directory operations.
import re  # Extracts JSON blocks from model outputs when needed.
from datetime import datetime  # Generates timestamps for log files.
import pandas as pd  # Imports the pandas library (aliased as pd) for handling and analyzing tabular data in DataFrames.
import numpy as np  # Imports NumPy (aliased as np), a library for fast numerical computations and array manipulations.
from tqdm import tqdm  # Imports tqdm, a progress bar utility that provides visual feedback for loops and long-running processes.
from langchain.chat_models import init_chat_model  # Imports a function from LangChain to initialize a chat-based large language model (LLM) interface.
from langchain_core.prompts import ChatPromptTemplate  # Imports a template class for creating structured prompts used in LLM interactions.
from langchain_core.output_parsers import PydanticOutputParser  # Parses JSON into Pydantic models.
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint  # Imports hosted Hugging Face endpoint wrappers.
from pydantic import BaseModel, Field  # Imports BaseModel (for defining structured data models) and Field (for specifying metadata and validation rules).
import krippendorff  # Imports the krippendorff library, used to compute Krippendorff’s alpha — a reliability measure for agreement among raters or annotators.

## Data loading

We encourage you to use the datasets available in SANE. The default code reuses the toy dataset from the previous notebook. 

In [ ]:
# Load CSV into a dataframe
data_url = "./data/srl_data_example.csv"
df = pd.read_csv(data_url)

Note that only the first 10 rows contain the anonymized text of the conversations. We will use these texts for the prompting experiments in this notebook.

In [ ]:
# Use these 10 rows to define test_ids and test_conversations
test_ids = df.id[:10].tolist()
test_conversations = df.conversation[:10].tolist()

## Local open-weights deployment in SANE with Ollama

We will use Ollama to run open-weights models locally. 
Ollama is installed by default in the SANE environment, but you will need to pull a model and start the local server to use it.

Example commands (run in a command line terminal):

- `ollama list` (to see available models)
- `ollama pull <model_name>` (e.g., `ollama pull qwen2.5:7b` to pull the 7B version of the Qwen2.5 model; this only needs to be done once per model)
- `ollama serve`  (starts the local server at http://localhost:11434; skip if already running)

Currently, we have the following models available locally via Ollama:
- qwen2.5:7b
- qwen2.5:14b
- qwen2.5-coder:7b
- qwen2.5-coder:14b
- gpt-oss:20b

The number after the colon indicates the number of parameters in the model (e.g., `qwen2.5:7b` has 7 billion parameters). Larger numbers indicate more parameters, hence bigger and more powerful (but often slower) models. The "coder" models are optimized for code generation, which may be useful for coding support and debugging.

In [ ]:
# Set up your model
model_name = "qwen2.5:7b"
temperature = 0  
max_tokens = 1000
seed = 123

# To prompt a self-hosted Ollama model, we simply point LangChain to the local server.
model = init_chat_model(
    model_name,
    model_provider="ollama",
    base_url="http://localhost:11434",
    temperature=temperature,
    max_tokens=max_tokens,
    seed=seed,
)

## Working with a single prompt

Let's start with the system prompt (i.e., high-level instruction to the model).

In [ ]:
# Define a system prompt that explains the task and scoring rubric
system_prompt = """
You are an expert in educational assessment and goal evaluation, with
specialized expertise in applying deductive coding schemes to score the quality
and content of student goals.

##TASK##
A university student was given a series of prompts, guiding them through the
process of setting and elaborating on an academic goal for the coming week. You
will be provided with the entire conversation including the prompts, and the
student answers. Your objective is to assess the specificity of of the student’s
goal on a scale of 0 to 2 based on the entire conversation.
"""

Use the prompt template module `ChatPromptTemplate` from langchain to create a prompt request with both **system** and **user** prompts.

In [ ]:
# Build a reusable prompt template with system + user parts
prompt_template = ChatPromptTemplate([
    ("system", system_prompt),
    ("user", "{conversation}"),
])

# Fill the template with the first conversation as a test case
single_prompt_request = prompt_template.invoke({"conversation": test_conversations[0]})

Check the prompt before prompting the model:

In [ ]:
# View the prepared message list (system + user)
single_prompt_request.to_messages()

Prompt the model and inspect the response!

In [ ]:
# Make the API call to get a single response
single_response = model.invoke(single_prompt_request)
print(single_response.content)

Voila! You have your first successful prompting interaction with an LLM API!

## Working with multiple prompts
Next, we go beyond a single prompt. Instead, we will work with **multiple prompts** at the same time.

**Tip**: Start with a small number of rows first to estimate time and (if using a paid API) cost to avoid surprises.

In [ ]:
# Iterate over the conversations and score them
multiple_responses = {}
for id, conversation in tqdm(zip(test_ids, test_conversations),
                             total=len(test_ids),
                             desc="Processing Requests"):
    prompt_request = prompt_template.invoke({"conversation": conversation})
    # Call the model and store the full response for each conversation by ID
    response = model.invoke(prompt_request)
    multiple_responses[id] = response.content

Inspect the responses!

In [ ]:
# Show an example response
print(multiple_responses['chat_2'])

## Using structured output with a single prompt

Use the `BaseModel` and `Field` classes from the `pydantic` package to specify the desired output format, and the model will return an output that matches it.

For example:

In [ ]:
# Define the expected structured output schema
class SpecificityFormat(BaseModel):
    goal_specificity: int = Field(
        description="Score for goal specificity. Only return an integer from 0 to 2",
    )
    reasoning: str = Field(description="The reasoning to justify the score")

**SURF AI Hub** and **OpenAI**

For SURF AI Hub, OpenAI and many other API providers, once you have defined the desired output format using the `BaseModel` and `Field` classes from the `pydantic` package, you can get structured output without having to parse it yourself.

In [ ]:
# Build a prompt for a single conversation
prompt_request = prompt_template.invoke({"conversation": test_conversations[0]})

if default_provider in ["openai", "surf_ai_hub"]:
    # OpenAI and SURF AI Hub can return structured outputs directly
    structured_model = model.with_structured_output(SpecificityFormat)
    single_structured_response = structured_model.invoke(prompt_request)
else:
    raise ValueError(f"Unsupported provider: {default_provider}")

In [ ]:
# Convert the structured response to a plain dict for inspection
dict(single_structured_response)

**Hugging Face**

Unlike the OpenAI API, Hugging Face hosted chat models do not support this automatically. Therefore, in addition, we need to explicitly **ask** the model for structured output in JSON format by adding instructions and an example to the prompt. Then, we need to **validate** the response, because the response might include extra text or even echo a schema. Finally, we can **parse** the JSON to get the structured output.

To this end, we define:
- `parser`: a `PydanticOutputParser` instance initialized with the expected output schema (`SpecificityFormat`).
- `extra_instructions`: a string containing additional explicit instructions to the model to return only JSON and not a schema.
- `build_structured_prompt(...)`: a function that adds `explicit instructions` to a prompt so the model knows to return only JSON.
- `parse_structured_response(...)`: a function that pulls out the JSON block and validates it against `SpecificityFormat`, raising a clear error if the model returns a schema or non‑JSON output.

In [ ]:
# Initialize a PydanticOutputParser with the expected output schema
parser = PydanticOutputParser(pydantic_object=SpecificityFormat)

# Define extra instructions to push the model to return only the JSON object
# and not a schema or extra text
extra_instructions = (
    "Return ONLY a JSON object with keys goal_specificity and reasoning.\n"
    "Do NOT return a schema.\n"
    "Example: {\"goal_specificity\": 2, \"reasoning\": \"...\"}"
    )

def build_structured_prompt(conversation: str,
                            system_prompt: str,
                            extra_instructions: str) -> ChatPromptTemplate:
    prompt_template_structured = ChatPromptTemplate.from_messages(
    [
        ("system", "{{ system_prompt }}\n\n{{ extra_instructions }}"),
        ("user", "{{ conversation }}"),
    ],
    template_format="jinja2",
).partial(
    system_prompt=system_prompt,
    extra_instructions=extra_instructions,
    )
    # Fill only the conversation at call time
    return prompt_template_structured.invoke({"conversation": conversation})

def parse_structured_response(text: str) -> SpecificityFormat:
    # Guard against the model echoing the schema instead of data
    if "\"properties\"" in text and "\"required\"" in text:
        raise ValueError("Model returned a schema, not data")
    # Prefer fenced JSON; fallback to any JSON object
    fenced = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL | re.IGNORECASE)
    if fenced:
        return parser.parse(fenced.group(1))
    inline = re.search(r"\{.*\}", text, re.DOTALL)
    if inline:
        return parser.parse(inline.group(0))
    raise ValueError("No JSON object found in the model output")

Try with a single prompt request.

In [ ]:
# Build a prompt for a single conversation
prompt_request = prompt_template.invoke({"conversation": test_conversations[0]})

if default_provider == "huggingface":
    # HF path: ask for JSON and then parse it into the schema
    prompt_request = build_structured_prompt(conversation=test_conversations[0],
                                             system_prompt=system_prompt,
                                             extra_instructions=extra_instructions)
    response = model.invoke(prompt_request)
    single_structured_response = parse_structured_response(response.content)
else:
    raise ValueError(f"Unsupported provider: {default_provider}")

In [ ]:
# Convert the structured response to a plain dict for inspection
dict(single_structured_response)

## Using structured output with multiple prompts

Being able to work with multiple prompts at the same time and obtain structured output will save you a substantial amount of time in research projects!

In [ ]:
# Score multiple conversations with structured output
multiple_structured_responses = {}
for id, conversation in tqdm(zip(test_ids, test_conversations),
                                     total=len(test_ids),
                                     desc="Processing Messages"):
    if default_provider == "openai":
        # OpenAI: structured output built in
        prompt_request = prompt_template.invoke({"conversation": conversation})
        structured_model = model.with_structured_output(SpecificityFormat)
        structured_response = structured_model.invoke(prompt_request)
    elif default_provider == "huggingface":
        # HF: request JSON and parse it
        prompt_request = build_structured_prompt(conversation=conversation,
                                                 system_prompt=system_prompt,
                                                 extra_instructions=extra_instructions)
        response = model.invoke(prompt_request)
        structured_response = parse_structured_response(response.content)
    else:
        raise ValueError(f"Unsupported provider: {default_provider}")
    multiple_structured_responses[id] = structured_response

Display all the structured responses:

In [ ]:
# Extract the scores from the structured responses
structured_scores = {id: resp.goal_specificity for id, resp in multiple_structured_responses.items()}
structured_scores

In [ ]:
# Extract the reasoning text from the structured responses
structured_reasonings = {id: resp.reasoning for id, resp in multiple_structured_responses.items()}
structured_reasonings

## Enhancing reproducibility: log prompts and decisions

To make your data collection/annotation reproducible, log each prompt, model configuration, and model output. The block below writes CSV and JSONL logs to the `logs/` folder with a timestamped filename.

As these logs include prompts and model outputs, make sure to remove or anonymize sensitive information before sharing logs.

We show below how to log the prompts and responses from the structured output experiment, but you can apply the same logic to log any other prompting experiment you do in this notebook or in your own projects.

In [ ]:
# Build a prompt + decision log
os.makedirs("logs", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_csv_path = f"logs/prompt_log_{timestamp}.csv"
log_jsonl_path = f"logs/prompt_log_{timestamp}.jsonl"

# Score multiple conversations with structured output
multiple_structured_responses = {} # Store the structured responses
prompt_logs = [] # Store the prompt logs
for id, conversation in tqdm(zip(test_ids, test_conversations),
                                     total=len(test_ids),
                                     desc="Processing Messages"):
    if default_provider == "openai":
        # OpenAI: structured output built in
        prompt_request = prompt_template.invoke({"conversation": conversation})
        structured_model = model.with_structured_output(SpecificityFormat)
        structured_response = structured_model.invoke(prompt_request)
    elif default_provider == "huggingface":
        # HF: request JSON and parse it
        prompt_request = build_structured_prompt(conversation=conversation,
                                                 system_prompt=system_prompt,
                                                 extra_instructions=extra_instructions)
        response = model.invoke(prompt_request)
        structured_response = parse_structured_response(response.content)
    else:
        raise ValueError(f"Unsupported provider: {default_provider}")
    multiple_structured_responses[id] = structured_response

    # Log the prompt, model config, and output
    messages = prompt_request.to_messages()
    prompt_text = "\n\n".join([f"{m.type.upper()}: {m.content}" for m in messages])
    prompt_logs.append({
        "request_id": id,
        "provider": default_provider,
        "model": model_name,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "max_new_tokens": max_new_tokens,
        "seed": seed,
        "prompt_text": prompt_text,
        "score": structured_scores.get(id),
        "reasoning": structured_reasonings.get(id),
    })

# Save logs to CSV and JSONL
prompt_log_df = pd.DataFrame(prompt_logs)
prompt_log_df.to_csv(log_csv_path, index=False)
prompt_log_df.to_json(log_jsonl_path, orient="records", lines=True)

# View the prompt log dataframe
prompt_log_df.head(3)

## Check annotation quality

Implement a handy function to calculate Krippendorff's Alpha (i.e., agreement) between two lists of specificity scores.

In [ ]:
def compute_krippendorff_alpha(x: list[int], y: list[int]):
  # Format data into a reliability matrix (rows=raters, cols=items)
  data_krippendorff = np.array([x, y])
  # Compute Krippendorff’s Alpha (interval metric)
  kripp_alpha = krippendorff.alpha(reliability_data=data_krippendorff,
                                   level_of_measurement='ordinal')
  return kripp_alpha

Let's check the agreement between the specificity scores we got from the LLM above and the human expert-coded specificity scores!

In [ ]:
# Compare agreement between expert and LLM ratings
expert_specificity_scores = df.expert_specificity_score[:10].tolist()
structured_llm_specificity_scores = list(structured_scores.values())
print("Krippendorff's Alpha:", compute_krippendorff_alpha(structured_llm_specificity_scores, expert_specificity_scores))

Not a great agreement score!

How about the agreement between the LLM specificity scores that already came with the dataset (i.e., column `score_specificity_llm`) and the human expert-coded scores?

Note that `score_specificity_llm` is based on prompts that were carefully engineered by Gabrielle.

In [ ]:
best_llm_specificity_scores = df.best_llm_specificity_score[:10].tolist()
print("Krippendorff's Alpha:", compute_krippendorff_alpha(best_llm_specificity_scores, expert_specificity_scores))

Wow! Much better!

## Exercise: Try different prompting techniques to get better results!

For example:

1. Improve clarity & specificity
2. Role-based prompting
3. Step-by-step reasoning (Chain-of-Thought Prompting)
4. Few-shot prompting
5. Output structuring
6. Self-consistency prompting

Use the previous `compute_krippendorff_alpha` function to check the LLM's annotation quality.

In [ ]:
# Let's write some code!